In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv()
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by checking whether the response contains any `function_call` items.

- It sends the current `messages` history to the model.
- If the model returns a `function_call`, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- After that turn, if `has_function_calls == False`, the loop breaks.
- Otherwise, it repeats and calls the model again with the updated history.

So the stop condition is simple: **no function calls in the latest response նշանակում է we're done.**


In [3]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

#provider = TracerProvider()
#provider.add_span_processor(
#    SimpleSpanProcessor(ConsoleSpanExporter())
#)
#trace.set_tracer_provider(provider)

#tracer = trace.get_tracer("llm-zoomcamp")

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [5]:
# Code needed for Q1 - Q6:

from rag_helper import RAGBase

class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("query", query)
            span.set_attribute("num_results_returned", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            span.set_attribute("model", self.model)
            span.set_attribute("response_length", len(response.output_text))
            span.set_attribute("input_tokens", response.usage.input_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            answer = super().rag(query)
            span.set_attribute("query", query)
            span.set_attribute("answer", answer)
            return answer

In [6]:
from dotenv import load_dotenv
load_dotenv()
from starter import index, client

rag_trace = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_trace.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and a `has_function_calls` flag.

- On each iteration, it sends the full `messages` history to the model.
- If the model returns any `function_call`, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If there are no function calls in that response, the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words: it keeps looping until the model returns a final answer with no more tool calls.


In [7]:
# Q1: How many spans does the trace produce?
# 3 Different spans for One Trace:
# Spans: "0xd265e9f205173abf", "0x2aa34e2f32b8c591", "0x8c2d1dffa2d354cf"

#Q2: How many input tokens do we see for the LLM call?
# 7111
#   },
#    "attributes": {
#        "model": "gpt-5.4-mini",
#        "response_length": 376,
#        "input_tokens": 7111
#   },

#Q3: For a typical query, roughly how long does the LLM call take?
# ~2 seconds
#    "start_time": "2026-07-20T20:44:56.973854Z",
#    "end_time": "2026-07-20T20:44:59.552525Z"

In [8]:
rag_trace = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_trace.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

Each iteration:
1. Sends the full `messages` history to the model.
2. Checks the response for any `function_call` items.
3. If there are function calls, it runs the tool, appends the tool result to `messages`, and loops again.
4. If there are no function calls, it breaks.

So the stopping condition is:

- **no function calls in the model’s response**

Then the loop ends and returns the final answer.


In [9]:
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784840365397572000,1784840365400344000,NaN,None,None
1,llm,1784840365401421000,1784840367454485000,7111.0,None,None
2,rag,1784840365397466000,1784840367456505000,NaN,None,None
3,search,1784840367515500000,1784840367517558000,NaN,None,None
4,llm,1784840367518942000,1784840369163366000,7111.0,None,None
5,rag,1784840367515449000,1784840369165379000,NaN,None,None


In [10]:
#Below results come from the df table generated from the above code.

#Q4: Which span names appear in the spans table?
# rag, search, llm

#Q5: Exluding the rag span, which span type take the most total time?
# llm, it takes about 1.5 seconds, while search takes about 0.4 seconds.

#Q6: How much do the input tokens vary across 4 runs of the same query?
# The input tokens are identical.

